In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from tokenizers import Tokenizer
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import pandas as pd
from torchmetrics.text.rouge import ROUGEScore
import random
from tqdm.auto import tqdm
import numpy as np
import os
import json
from copy import deepcopy

## Архитектура LSTM + Attention

In [2]:
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size)
        self.U = nn.Linear(2 * hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, enc_outputs, enc_projection, hidden, mask):
        """
        enc_outputs: [batch, seq_len1, 2 * hidden_size] 
        hidden: [batch, hidden_size]
        mask: [batch, seq_len1]
        """
        hidden = hidden.unsqueeze(1).expand(hidden.shape[0],       # [batch, seq_len1, hidden_size]
                                            enc_outputs.shape[1], 
                                            hidden.shape[1])  
        score = torch.tanh(self.W(hidden) + enc_projection)        # [batch, seq_len1, hidden_size]
        score = self.v(score).squeeze(2)                           # [batch, seq_len1]
        score = score.masked_fill(mask == False, -1e9)             # [batch, seq_len1]
        attention = self.softmax(score)                            # [batch, seq_len1]
        context = torch.bmm(attention.unsqueeze(1), enc_outputs)   # [batch, 1, 2 * hidden_size]
        return context


class Seq2Seq(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=2, embedding=None, vocab_size=None, dropout=0.4):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.vocab_size = vocab_size
        
        if embedding:
            self.embedding = embedding
        else:
            self.embedding = nn.Embedding(vocab_size, input_size)
        
        self.encoder = nn.LSTM(input_size, hidden_size, 
                               num_layers=num_layers, 
                               batch_first=True, 
                               bidirectional=True,
                               dropout=dropout)
        self.decoder = nn.LSTM(input_size + (2 * hidden_size), hidden_size, 
                               num_layers=num_layers, 
                               batch_first=True, 
                               dropout=dropout)
        self.adapter_h = nn.Linear(2 * hidden_size, hidden_size)
        self.adapter_c = nn.Linear(2 * hidden_size, hidden_size)
        self.fc_out = nn.Linear(hidden_size * 3, vocab_size)
        self.attention = Attention(hidden_size)
        
    def encode(self, enc_x, enc_len):
        """
        enc_x: Long - [batch, seq_len1] 
        enc_len: Long - [batch]
        """
        enc_x = self.embedding(enc_x)  # [batch, seq_len1, input_size] 

        packed_enc_x = nn.utils.rnn.pack_padded_sequence(            
            enc_x, enc_len, batch_first=True, enforce_sorted=False
        )
        # outputs: PackedSequence (в распакованном виде было бы [batch, seq_len1, 2 * hidden_size])
        # h: [2 * num_layers, batch, hidden_size] (2 т.к. bidirectional: слои forward и backward)
        # c: [2 * num_layers, batch, hidden_size]
        outputs, (h, c) = self.encoder(packed_enc_x)                      
        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs,            # [batch, seq_len1, 2 * hidden_size]
                                                      batch_first=True)
        
        decoder_init_h = torch.cat([h[::2, :, :], h[1::2, :, :]], dim=2)  # [num_layers, batch, 2 * hidden_size]
        decoder_init_c = torch.cat([c[::2, :, :], c[1::2, :, :]], dim=2)  # [num_layers, batch, 2 * hidden_size]
        decoder_init_h = torch.tanh(self.adapter_h(decoder_init_h))       # [num_layers, batch, hidden_size]
        decoder_init_c = torch.tanh(self.adapter_c(decoder_init_c))       # [num_layers, batch, hidden_size]
        
        return outputs, decoder_init_h, decoder_init_c

    def create_mask(self, enc_len, max_len, device):
        batch_size = enc_len.shape[0]
        mask = torch.arange(max_len).expand(batch_size, max_len) < enc_len.unsqueeze(1)
        
        return mask.to(device)
    
    def forward(self, enc_x, dec_x, enc_len, teacher_forcing_ratio=0.4): 
        """ 
        enc_x: Long - [batch, seq_len1] 
        dec_x: Long - [batch, seq_len2] 
        enc_len: Long - [batch] 
        """
        # enc_outputs: [batch, seq_len1, 2 * hidden_size] 
        # decoder_init_h: [num_layers, batch, hidden_size] 
        # decoder_init_c: [num_layers, batch, hidden_size] 
        enc_outputs, decoder_init_h, decoder_init_c = self.encode(enc_x, enc_len) 
        enc_projection = self.attention.U(enc_outputs) 
        
        decoder_input = self.embedding(dec_x)                        # [batch, seq_len2, input_size] 
        mask = self.create_mask(enc_len, enc_outputs.shape[1],       # [batch, seq_len1]
                                device=enc_x.device)     

        batch_size = dec_x.shape[0]
        trg_len = dec_x.shape[1]

        outputs = torch.zeros(batch_size, trg_len, self.vocab_size, device=enc_x.device)
        hidden, cell = decoder_init_h, decoder_init_c
        dec_input = decoder_input[:, :1]                          # [batch, 1, input_size] 
        
        for t in range(trg_len):
            context = self.attention(enc_outputs, enc_projection, hidden[-1], mask)  # [batch, 1, 2 * hidden_size] 
            dec_input = torch.cat([dec_input, context], dim=2)
            output, (hidden, cell) = self.decoder(dec_input, (hidden, cell)) 
            output = torch.cat([output, context], dim=2)
            output = self.fc_out(output.squeeze(1))                  # [batch, vocab_size]
            outputs[:, t] = output

            if t < trg_len - 1:
                top1 = output.argmax(1)                                  # [batch]
                if random.random() < teacher_forcing_ratio:
                    dec_input = decoder_input[:, t+1:t+2]
                else:
                    dec_input = self.embedding(top1.unsqueeze(1))
        
        return outputs 
    
    def generate(self, enc_x, enc_len, sos_idx=1, eos_idx=2, max_len=100):
        """
        enc_x: Long - [batch = 1, seq_len1] for inference
        enc_len: Long - [batch = 1]
        """
        self.eval()
        with torch.no_grad():
            device = enc_x.device
            enc_outputs, current_h, current_c = self.encode(enc_x, enc_len)         
            mask = self.create_mask(enc_len, enc_outputs.shape[1], device=device)  # [batch=1, seq_len1]
            encoder_projection = self.attention.U(enc_outputs)
            decoder_idx = torch.tensor([[sos_idx]], device=device)
            
            outputs = []
            for _ in range(max_len):
                dec_input = self.embedding(decoder_idx)                            # [batch=1, 1, input_size]
                context = self.attention(enc_outputs, encoder_projection, current_h[-1], mask)  # [batch=1, 1, 2 * hidden_size] 
                dec_input = torch.cat([dec_input, context], dim=2)                 # [batch=1, 1, input_size + (2 * hidden_size)] 
                output, (current_h, current_c) = self.decoder(dec_input, (current_h, current_c))
                output = torch.cat([output, context], dim=2)
                output = self.fc_out(output)                                       # [batch=1, 1, vocab_size]
                decoder_idx = torch.argmax(output, 2)                              # [batch=1, 1] 
                if decoder_idx.item() == eos_idx:
                    break
                outputs.append(decoder_idx[0])

            if len(outputs) == 0:
                outputs = torch.tensor([], device=device, dtype=torch.long)
            else:
                outputs = torch.cat(outputs)
        return outputs

### Аугментации

In [4]:
class FastSynonymAug:
    def __init__(self, dict_path="synonyms_dict.json", aug_p=0.4):
        self.aug_p = aug_p
        with open(dict_path, 'r', encoding='utf-8') as f:
            self.synonyms = json.load(f)

    def __call__(self, text):
        if not text: return text
        
        words = text.split()
        new_words = [
            self._replace_word(w) if random.random() < self.aug_p else w 
            for w in words
        ]
        
        return " ".join(new_words)

    def _replace_word(self, word):
        clean_word = word.lower().strip('.,!?:;') 

        if clean_word in self.synonyms:
            syn = random.choice(self.synonyms[clean_word])
            if word[0].isupper():
                syn = syn.capitalize()
            if not word[-1].isalnum():
                syn += word[-1]
            return syn
        return word


class FastWordSwapAug:
    def __init__(self, aug_p=0.15):
        self.aug_p = aug_p

    def __call__(self, text):
        words = text.split()
        length = len(words)
        
        if length < 2: return text
        
        if random.random() < self.aug_p:
            idx = random.randint(0, length-2)
            words[idx], words[idx+1] = words[idx+1], words[idx]
            return " ".join(words)
            
        return text


class EfficientPipeline:
    def __init__(self, synonyms_path):
        self.aug_synonym = FastSynonymAug(synonyms_path, aug_p=0.4)
        self.aug_swap = FastWordSwapAug(aug_p=0.3)
        
    def __call__(self, text, p=[0.5, 0.5]):
        if random.random() < p[0]:
            text = self.aug_synonym(text)

        if random.random() < p[1]:
            text = self.aug_swap(text)
            
        return text

In [5]:
class TextSummarizationDataset(Dataset):
    def __init__(self, data, tokenizer, augment=True, max_text_len=500):
        self.data = data
        self.tokenizer = tokenizer
        self.augment = augment
        self.max_text_len = max_text_len
        
        if augment:
            self.augs = EfficientPipeline(Parameters.synonyms_path)

    def __getitem__(self, indx):
        data = self.data.iloc[indx]
        text, summary = data.iloc[0], data.iloc[1]

        if self.augment:
            text = self.augs(text)
            
        text_ids = self.tokenizer.encode(text).ids
        summary_ids = self.tokenizer.encode(summary).ids

        text_ids = text_ids[:self.max_text_len]

        return torch.tensor(text_ids), torch.tensor(summary_ids), len(text_ids)

    def __len__(self):
        return len(self.data)

In [6]:
@dataclass 
class Parameters:
    train_path: str = "gazeta-dataset/train.csv"
    valid_path: str = "gazeta-dataset/valid.csv"
    test_path: str =  "gazeta-dataset/test.csv"
    tokenizer_path: str = "summarization-tokenizer/my_tokenizer.json"
    synonyms_path: str = "synonyms-dict/synonyms_dict.json"
    batch_size: int = 24
    num_epochs: int = 25
    vocab_size: int = 30000
        
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Инициализация объектов датасета

In [7]:
tokenizer = Tokenizer.from_file(Parameters.tokenizer_path)

In [8]:
tokenizer.get_vocab_size()

30000

In [9]:
train_data = pd.read_csv(Parameters.train_path)
valid_data = pd.read_csv(Parameters.valid_path)
test_data = pd.read_csv(Parameters.test_path)

In [10]:
train_data.shape

(62793, 2)

In [11]:
valid_data.shape

(6369, 2)

In [12]:
test_data.shape

(4964, 2)

In [13]:
train_dataset = TextSummarizationDataset(train_data, tokenizer, augment=True)
valid_dataset = TextSummarizationDataset(valid_data, tokenizer, augment=False)
test_dataset = TextSummarizationDataset(test_data, tokenizer, augment=False)

In [14]:
def collate_fn(batch):
    text, summary, text_lengths = zip(*batch)

    pad_text = nn.utils.rnn.pad_sequence(text, batch_first=True, padding_value=0)
    pad_summary = nn.utils.rnn.pad_sequence(summary, batch_first=True, padding_value=0)

    return pad_text, pad_summary, torch.tensor(text_lengths)

In [25]:
train_loader = DataLoader(
    train_dataset, 
    batch_size=Parameters.batch_size, 
    shuffle=True, 
    num_workers=4, 
    drop_last=True, 
    collate_fn=collate_fn, 
    pin_memory=True,
    persistent_workers=True
)

valid_loader = DataLoader(
    valid_dataset, 
    batch_size=1, 
    shuffle=False, 
    num_workers=4, 
    drop_last=False, 
    collate_fn=collate_fn,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=1, 
    shuffle=False, 
    num_workers=4, 
    drop_last=False, 
    collate_fn=collate_fn,
    pin_memory=True,
    persistent_workers=True
)

In [17]:
model = Seq2Seq(
    input_size=300, 
    hidden_size=300, 
    num_layers=2, 
    vocab_size=Parameters.vocab_size
).to(device)

Убирает `weight_decay` (L2-регуляризацию) со слоев смещении (bias).

In [19]:
def get_grouped_params(model, weight_decay):
    decay_params = []
    no_decay_params = []
    
    for name, param in model.named_parameters():
        if "bias" in name or "embedding" in name:
            no_decay_params.append(param)
        else:
            decay_params.append(param)
            
    return [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]

In [20]:
grouped_params = get_grouped_params(model, weight_decay=5e-3)
optimizer = optim.AdamW(grouped_params, lr=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Parameters.num_epochs, eta_min=1e-5)
criterion = nn.CrossEntropyLoss(ignore_index=0).to(device)
rouge = ROUGEScore()

In [21]:
sos_id = tokenizer.token_to_id("<SOS>")
eos_id = tokenizer.token_to_id("<EOS>")

## Дообучение модели

In [22]:
best_model = None
best_score = 0

for epoch in range(Parameters.num_epochs):
    model.train()
    sum_loss = 0
    d = 0
    for pad_text, pad_summary, text_lengths in tqdm(train_loader):
        pad_text = pad_text.to(device)
        pad_summary = pad_summary.to(device)

        target = pad_summary[:, 1:]
        dec_input = pad_summary[:, :-1]

        pred = model(pad_text, dec_input, text_lengths)
        pred = pred.reshape(-1, pred.shape[-1])
        target = target.reshape(-1)

        optimizer.zero_grad()
        loss = criterion(pred, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()

        sum_loss += loss.item()
        d += 1

    model.eval() 
    rouge.reset() 

    with torch.no_grad(): 
        for i, (pad_text, pad_summary, text_lengths) in enumerate(valid_loader):
            pad_text = pad_text.to(device)
            generated_ids = model.generate(pad_text, text_lengths, max_len=70)
            
            if isinstance(generated_ids, torch.Tensor):
                generated_ids = generated_ids.tolist()
            
            pred_str = tokenizer.decode(generated_ids, skip_special_tokens=True)
            target_ids = pad_summary[0].tolist()
            label_str = tokenizer.decode(target_ids, skip_special_tokens=True)
    
            rouge.update([pred_str], [label_str])

            if i == 0:
                print(f"Target: {label_str}")
                print(f"Pred:   {pred_str}")
            
    result = rouge.compute()
    rouge1 = result['rouge1_fmeasure'].item()
    rouge2 = result['rouge2_fmeasure'].item()
    rougeL = result['rougeL_fmeasure'].item()
    
    print(f"ROUGE-1: {rouge1:.4f} (совпадение слов)")
    print(f"ROUGE-2: {rouge2:.4f} (совпадение пар слов)")
    print(f"ROUGE-L: {rougeL:.4f} (структура предложений)")
    print(f"Loss: {sum_loss / d:.4f}")

    scheduler.step()

    if rouge2 > best_score:
        best_model = deepcopy(model.state_dict())
        best_score = rouge2

    torch.save(model.state_dict(), f"Epoch: {epoch + 1} Metric: {rouge2:4f}.pth")

  0%|          | 0/2616 [00:00<?, ?it/s]
Target: в уходящем году инфляция в россии находится на историческом минимуме. в следующем году ожидается, что она также будет минимальной. однако стоимость ряда продуктов и напитков в 2020 году может вырасти гораздо выше инфляции. это касается молочных продуктов. вырастет в цене водка, коньяк и вино, продолжит дорожать гречка.
Pred:   рост цен на инфляции в россии в россии в в в на на%, прогнозируют эксперты. в этом году в на на на на на на на на на на на и в на на и в на на и и в в в в в в и в в в
ROUGE-1: 0.0436 (совпадение слов)
ROUGE-2: 0.0131 (совпадение пар слов)
ROUGE-L: 0.0432 (структура предложений)
Loss: 5.6027
  0%|          | 0/2616 [00:00<?, ?it/s]
Target: в уходящем году инфляция в россии находится на историческом минимуме. в следующем году ожидается, что она также будет минимальной. однако стоимость ряда продуктов и напитков в 2020 году может вырасти гораздо выше инфляции. это касается молочных продуктов. вырастет в цене водка, 